# 第3週課題：データで見つけた、小さな発見
## テーマ：東京の夏の気温を調べる

Open-MeteoのAPIから東京の過去の気象データを取得し、2016〜2025年の夏（6〜8月）の日最高気温を年ごとに比較する。

**このNotebookは、Pythonに最初から入っている標準機能だけを使います。** `pandas`、`matplotlib`、追加インストールは不要です。APIに接続できるインターネット環境は必要です。

## 1. APIからデータを取得する

次のセルを実行してください。もしエラーが出た場合は、エラーメッセージを確認してください。

In [1]:
import json
import urllib.request
from datetime import date

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=35.6762&longitude=139.6503"
    "&start_date=2016-01-01&end_date=2025-12-31"
    "&daily=temperature_2m_max"
    "&timezone=Asia%2FTokyo"
)

try:
    with urllib.request.urlopen(url, timeout=30) as response:
        data = json.loads(response.read().decode("utf-8"))
    print("APIからデータを取得できました。")
    print("取得した日数：", len(data["daily"]["time"]))
except Exception as e:
    print("APIからデータを取得できませんでした。")
    print("エラー内容：", e)
    print("学校のネットワークなどで接続が制限されている可能性があります。")

APIからデータを取得できました。
取得した日数： 3653


## 2. 夏（6〜8月）の気温を年ごとに集計する

各年について、夏の日最高気温の平均と、最高気温が35℃以上だった日数を計算します。

In [2]:
# API取得が成功していることを確認してから実行
daily = data["daily"]
temps_by_year = {}
hot_days_by_year = {}

for day, temp in zip(daily["time"], daily["temperature_2m_max"]):
    month = int(day[5:7])
    year = int(day[:4])

    # 夏は6月・7月・8月
    if month in (6, 7, 8) and temp is not None:
        temps_by_year.setdefault(year, []).append(temp)
        if temp >= 35:
            hot_days_by_year[year] = hot_days_by_year.get(year, 0) + 1
        else:
            hot_days_by_year.setdefault(year, 0)

print("年 | 夏の日最高気温の平均(℃) | 最高気温35℃以上の日数")
print("-" * 58)

for year in sorted(temps_by_year):
    temps = temps_by_year[year]
    avg_temp = sum(temps) / len(temps)
    hot_days = hot_days_by_year.get(year, 0)
    print(f"{year} | {avg_temp:.2f} | {hot_days}")

年 | 夏の日最高気温の平均(℃) | 最高気温35℃以上の日数
----------------------------------------------------------
2016 | 27.58 | 1
2017 | 28.90 | 1
2018 | 29.39 | 2
2019 | 28.01 | 0
2020 | 28.95 | 5
2021 | 28.60 | 1
2022 | 29.65 | 5
2023 | 31.42 | 19
2024 | 30.98 | 19
2025 | 32.11 | 28


## 3. データから見つけたことを確認する

次のセルで、平均気温が最も高かった年と低かった年を確認します。

In [3]:
yearly_avg = {
    year: sum(temps) / len(temps)
    for year, temps in temps_by_year.items()
}

if yearly_avg:
    hottest_year = max(yearly_avg, key=yearly_avg.get)
    coolest_year = min(yearly_avg, key=yearly_avg.get)

    print(f"夏の日最高気温の平均が最も高かった年：{hottest_year}年（{yearly_avg[hottest_year]:.2f}℃）")
    print(f"夏の日最高気温の平均が最も低かった年：{coolest_year}年（{yearly_avg[coolest_year]:.2f}℃）")

    first = [yearly_avg[y] for y in yearly_avg if 2016 <= y <= 2020]
    second = [yearly_avg[y] for y in yearly_avg if 2021 <= y <= 2025]
    if first and second:
        first_avg = sum(first) / len(first)
        second_avg = sum(second) / len(second)
        print(f"2016〜2020年の平均：{first_avg:.2f}℃")
        print(f"2021〜2025年の平均：{second_avg:.2f}℃")
        print(f"後半と前半の差：{second_avg - first_avg:+.2f}℃")
else:
    print("集計データがありません。最初のセルが成功したか確認してください。")

夏の日最高気温の平均が最も高かった年：2025年（32.11℃）
夏の日最高気温の平均が最も低かった年：2016年（27.58℃）
2016〜2020年の平均：28.57℃
2021〜2025年の平均：30.55℃
後半と前半の差：+1.98℃


## 4. 課題の記録（実行結果を確認して完成させる）

**何を調べ、何が分かったか**  
東京の2016〜2025年の夏の日最高気温を調べ、年ごとの平均と、最高気温が35℃以上の日数を比較した。上の集計結果から、平均気温が最も高い年・低い年を確認できた。実行後、表示された具体的な数値を使って、自分が気づいた点を追記する。

**元データと照らして確かめた値**  
APIから取得した日別の最高気温を使い、6〜8月だけを集計した。各年の平均気温は、その年の夏のデータを合計して日数で割って求めた。35℃以上の日を数えて猛暑日数とした。

**このデータだけでは分からないこと**  
東京の1地点だけのデータなので、日本全体の気温の傾向とは限らない。また、10年間のデータだけでは、気温変化の原因を断定できない。都市化や年ごとの天候の違いなども関係する可能性がある。

**出典URL**  
Open-Meteo Historical Weather API：  
https://open-meteo.com/en/docs/historical-weather-api

**データ取得先**  
https://archive-api.open-meteo.com/v1/archive

**対象地点・期間**  
東京（緯度35.6762、経度139.6503）、2016年1月1日〜2025年12月31日

**取得日**  
2026

**AIの利用**  
コードの作成と説明文の下書きに生成AIを使用した。APIから取得したデータを実行し、表示された結果を確認した。